**Data Preprocessing**
Goal is to prepare our raw data for later processing

In [9]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import pandas as pd
import os
from src.preprocessing import ensure_directories, initial_cleaning, normalize_text, process_and_tokenize, print_reduction_rates, parallel_process, wrapper_normalize, wrapper_tokenize

ensure_directories()

file_path = "../data/raw/995,000_rows.csv"

if os.path.exists(file_path):
    df_sample = pd.read_csv(file_path, low_memory=False, dtype={'Unnamed: 0': str, 'id': str})
else:
    print("File not found")

print(f"DataFrame has {df_sample.shape[0]} rows and {df_sample.shape[1]} columns")

print("--- Columns and DTypes ---")
print(df_sample.dtypes)

print("\n--- Missing values per column ---")
print(df_sample.isnull().sum())

df_sample.head()

DataFrame has 995000 rows and 17 columns
--- Columns and DTypes ---
Unnamed: 0              str
id                      str
domain                  str
type                    str
url                     str
content                 str
scraped_at              str
inserted_at             str
updated_at              str
title                   str
authors                 str
keywords            float64
meta_keywords           str
meta_description        str
tags                    str
summary             float64
source                  str
dtype: object

--- Missing values per column ---
Unnamed: 0               1
id                       7
domain                  11
type                 47786
url                     11
content                 12
scraped_at              13
inserted_at             13
updated_at              13
title                 8606
authors             442757
keywords            995000
meta_keywords        38790
meta_description    525106
tags                764081
su

,Unnamed: 0,id,domain,type,url,content,scraped_at,inserted_at,updated_at,title,authors,keywords,meta_keywords,meta_description,tags,summary,source
0,732,7444726.0,nationalreview.com,political,http://www.nationalreview.com/node/152734/%E2%...,Plus one article on Google Plus\n\n(Thanks to ...,2017-11-27T01:14:42.983556,2018-02-08 19:18:34.468038,2018-02-08 19:18:34.468066,Iran News Round Up,NaN,NaN,"['National Review', 'National Review Online', ...",NaN,NaN,NaN,NaN
1,1348,6213642.0,beforeitsnews.com,fake,http://beforeitsnews.com/economy/2012/06/the-c...,The Cost Of The Best Senate Banking Committee ...,2017-11-27T01:14:08.7454,2018-02-08 19:18:34.468038,2018-02-08 19:18:34.468066,The Cost Of The Best Senate Banking Committee ...,NaN,NaN,[''],NaN,NaN,NaN,NaN
2,7119,3867639.0,dailycurrant.com,satire,http://dailycurrant.com/2016/01/18/man-awoken-...,Man Awoken From 27-Year Coma Commits Suicide A...,2017-11-27T01:14:21.395055,2018-02-07 23:39:33.852671,2018-02-07 23:39:33.852696,Man Awoken From 27-Year Coma Commits Suicide A...,NaN,NaN,[''],NaN,NaN,NaN,NaN
3,1518,9560791.0,nytimes.com,reliable,https://query.nytimes.com/gst/fullpage.html?re...,WHEN Julia Geist was asked to draw a picture o...,2018-02-11 00:46:42.632962,2018-02-11 00:14:20.346838,2018-02-11 00:14:20.346871,Opening a Gateway for Girls to Enter the Compu...,NaN,NaN,"['Computers and the Internet', 'Women and Girl...",WHEN Julia Geist was asked to draw a picture o...,NaN,NaN,nytimes
4,9345,2059625.0,infiniteunknown.net,conspiracy,http://www.infiniteunknown.net/2011/09/14/100-...,– 100 Compiled Studies on Vaccine Dangers (Act...,2017-11-10T11:18:44.524042,2018-02-07 23:39:33.852671,2018-02-07 23:39:33.852696,100 Compiled Studies on Vaccine Dangers – Infi...,NaN,NaN,[''],NaN,"Lymphoma, Hepatitis B, Immune System, Health, ...",NaN,NaN


***From the initial overview of the raw data the following stands out:**
- `keywords` & `summary` are competely empty -> we should drop.
- There is quite a few missing entries in `type`(47.786). Since this is our target value, in terms of modelling and evaluating this is useless -> we should drop.
- `Unnamed: 0`, `id`, `inserted_at` and `updated_at` carry little to no predictive power -> we should drop.
- We keep `scraped_at` for when we split for train/validate/test. This is to prevent data leakage. However, we need to convert this do datetime. OBS. since there is a T (some ISO standard) we need to handle that before.
- For `authors`, `title` and we replace group missing entries by a "missing category" (Feature Engineering)
- Since `domain` only has 11 missing entries we simply drop. Most likely these will be on the same rows as a missing `type`.
- We also drop 12 rows with missing `content`

In [10]:
df_cleaned = initial_cleaning(df_sample)
print(df_cleaned.head())
print("\n--- Missing values per column ---")
print(df_cleaned.isnull().sum())

Cleaning done: Start=995000, End=947213, Dropped=47787 rows.
          id               domain        type  \
0  7444726.0   nationalreview.com   political   
1  6213642.0    beforeitsnews.com        fake   
2  3867639.0     dailycurrant.com      satire   
3  9560791.0          nytimes.com    reliable   
4  2059625.0  infiniteunknown.net  conspiracy   

                                                 url  \
0  http://www.nationalreview.com/node/152734/%E2%...   
1  http://beforeitsnews.com/economy/2012/06/the-c...   
2  http://dailycurrant.com/2016/01/18/man-awoken-...   
3  https://query.nytimes.com/gst/fullpage.html?re...   
4  http://www.infiniteunknown.net/2011/09/14/100-...   

                                             content  \
0  Plus one article on Google Plus\n\n(Thanks to ...   
1  The Cost Of The Best Senate Banking Committee ...   
2  Man Awoken From 27-Year Coma Commits Suicide A...   
3  WHEN Julia Geist was asked to draw a picture o...   
4  – 100 Compiled Studies o

This output looks good. Everything is as expected.

We still need to decide what to do with the meta-data. Most likely since `meta_description`, `tags`, and `source` have such a high number of missing entries, we will just drop.
But `meta_keywords` might be useful in part 2 of the project

In [11]:
test_article = """
    BREAKING NEWS!!! Check out https://fakenews.com/article.
    Contact us at spam@email.com.
    This happened on 2023-10-25 and cost 1000000 dollars.
"""

print(normalize_text(test_article))

breaking news check out urltoken contact us at emailtoken this happened on datetoken and cost numtoken dollars


In [12]:
print("starting normalization of 'content' and 'title' columns using parallel processing")
df_cleaned = parallel_process(df_cleaned, wrapper_normalize)

df_cleaned[['title', 'content']].head()

starting normalization of 'content' and 'title' columns using parallel processing


,title,content
0,iran news round up,plus one article on google plus thanks to ali ...
1,the cost of the best senate banking committee ...,the cost of the best senate banking committee ...
2,man awoken from numtoken year coma commits sui...,man awoken from numtoken year coma commits sui...
3,opening a gateway for girls to enter the compu...,when julia geist was asked to draw a picture o...
4,numtoken compiled studies on vaccine dangers i...,numtoken compiled studies on vaccine dangers a...


In [13]:
import random

random_indices = random.sample(range(len(df_cleaned)), 3)

for idx in random_indices:
    print(f"--- Artikel {idx} ---")

    print(df_cleaned['content'].iloc[idx][:500])
    print("\n")

--- Artikel 227283 ---
the los angeles times formally retracted on monday its investigative report about a numtoken attack on the rapper tupac shakur after concluding that a prison inmate had fabricated much of the evidence cited in the article the article published online on march numtoken and in the newspaper two days later suggested that associates of the rapper sean combs were involved in a shooting of mr shakur a numtoken word correction on the front of the calendar section stated that the report relied heavily o


--- Artikel 895670 ---
an obituary of the oscar winning film composer elmer bernstein on aug numtoken misspelled the surname of a film composer he admired that composer was miklos rozsa not rosza


--- Artikel 378438 ---
you are like a prison doctor treating a victim of torture making the prisoner healthy to be interrogated and tortured again hassan al zeyada who has spent decades counseling fellow residents of the gaza strip who suffer from psychological trauma




This print of three random articles shows what we have succeeded in normalizing the data

In [14]:
print("Running NLTK Tokenization, Stopwords and Stemming using parallel processing")
df_cleaned = parallel_process(df_cleaned, wrapper_tokenize)
df_cleaned.head()

Running NLTK Tokenization, Stopwords and Stemming using parallel processing


,id,domain,type,url,content,scraped_at,title,authors,meta_keywords,meta_description,tags,source,content_processed
0,7444726.0,nationalreview.com,political,http://www.nationalreview.com/node/152734/%E2%...,plus one article on google plus thanks to ali ...,2017-11-27 01:14:42.983556+00:00,iran news round up,unknown_author,"['National Review', 'National Review Online', ...",NaN,NaN,NaN,plus one articl googl plus thank ali alfoneh a...
1,6213642.0,beforeitsnews.com,fake,http://beforeitsnews.com/economy/2012/06/the-c...,the cost of the best senate banking committee ...,2017-11-27 01:14:08.745400+00:00,the cost of the best senate banking committee ...,unknown_author,[''],NaN,NaN,NaN,cost best senat bank committe jp morgan buy $$...
2,3867639.0,dailycurrant.com,satire,http://dailycurrant.com/2016/01/18/man-awoken-...,man awoken from numtoken year coma commits sui...,2017-11-27 01:14:21.395055+00:00,man awoken from numtoken year coma commits sui...,unknown_author,[''],NaN,NaN,NaN,man awoken numtoken year coma commit suicid le...
3,9560791.0,nytimes.com,reliable,https://query.nytimes.com/gst/fullpage.html?re...,when julia geist was asked to draw a picture o...,2018-02-11 00:46:42.632962+00:00,opening a gateway for girls to enter the compu...,unknown_author,"['Computers and the Internet', 'Women and Girl...",WHEN Julia Geist was asked to draw a picture o...,NaN,nytimes,julia geist ask draw pictur comput scientist l...
4,2059625.0,infiniteunknown.net,conspiracy,http://www.infiniteunknown.net/2011/09/14/100-...,numtoken compiled studies on vaccine dangers a...,2017-11-10 11:18:44.524042+00:00,numtoken compiled studies on vaccine dangers i...,unknown_author,[''],NaN,"Lymphoma, Hepatitis B, Immune System, Health, ...",NaN,numtoken compil studi vaccin danger activist p...


In [15]:
output_path = '../data/processed/995K_cleaned.csv'
df_cleaned.to_csv(output_path, index=False)

In [16]:
# Calculate reduction rates
print_reduction_rates()


--- Final reduction rates ---
Original vocabulary: 1,670,237 unique tokens
After stopwords: 1,670,086 tokens (0.01% reduction)
After stemming: 1,430,252 tokens (14.36% reduction from previous)
Total reduction: 14.37%
